In [ ]:
# Step 1: Import PyTorch and Check Devic
import torch

print(f"Using torch {torch.__version__}")
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device = {device}")


In [ ]:
train_dir = "data/train"
test_dir = "data/val"

In [ ]:
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 
0.224, 0.225))
])
test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 
0.224, 0.225))
])

In [ ]:
#  Step 4: Load the Imagenette Dataset

train_data = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_data = datasets.ImageFolder(root=test_dir, transform=test_transform)

In [ ]:
# Step 5: Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
# Step 6: Inspect the Dataset
print(f"Training data contains {len(train_data)} images")
print(f"Testing data contains {len(test_data)} images")

class_names = train_data.classes
print("Classes:", class_names)

class_names_idx = train_data.class_to_idx
print("Class to index mapping:", class_names_idx)


In [ ]:
# Step 7: Import and Instantiate the Model
from model import CustomCNN

num_classes = len(train_data.classes)
model = CustomCNN(num_classes).to(device)


In [ ]:
# Step 8: Print a Model Summary
from torchinfo import summary
model.to(device)

summary(model=model,
        input_size=(32, 3, 128, 128),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

In [ ]:
# Step 9: Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

In [ ]:
torch.manual_seed(42)
if device == "cuda":
    torch.cuda.manual_seed(42)

from timeit import default_timer as timer
from trainNN import train

start_time = timer()
results = train(
    model=model,
    train_dataloader=train_loader,
    test_dataloader=test_loader,
    optimizer=optimizer,
    loss_fn=criterion,
    epochs=5,
    device=device
)
end_time = timer()
print(f"[INFO] Total training time: {end_time - start_time:.2f} seconds")


In [ ]:
# Step 11: Plot Loss and Accuracy Curves
from helper_functions import plot_loss_curves
plot_loss_curves(results)